# Preprocessing

<div style="text-align: justify">
The following notebook is dedicated to preprocessing for the  <b> Tau Supersymmetry </b> search analysis. Preprocessing involves handling .ROOT files for use in both cut-and-count- and machine learning-based analyses. A general data processing workflow is structured as follows:
</div>

<img src="../assets/data_processing.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis configuration |
| Resolve | `analysis.resolve_samples` | Build sample lists from config |
| Process | `processor.process_samples` | ROOT → cuts → weights per campaign |
| Merge | `merger.merge_backgrounds` | Group backgrounds by strategy |
| Combine | `merger.combine_background_signal` | Combine with signal samples |
| Label | `merger.assign_class` | Assign integer class labels |
| Save | `io.save_samples` | Write parquet files |

The same pipeline is available as a CLI via `python preprocess.py` or `make preprocess`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data I/O:
* [UpROOT](https://uproot.readthedocs.io/en/latest/)
* [Awkward Array](https://awkward-array.readthedocs.io/en/latest/)

Serialization:
* [Apache Parquet](https://parquet.apache.org/)

### Notebook

Activating autoreload of imported modules.

In [1]:
%load_ext autoreload
%autoreload 2

Initializing the project root.

In [2]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [3]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

Unessential warnings suppressed.
ATLAS style applied with LaTeX.


## Overview

List of available analyses.

<br>

<hr>

Run 3:
* <b> release </b>: R24
* <b> analysis base </b>: '24.2.28'
* <b> campaigns </b>: ['MC23a', 'MC23c']
* <b> architecture </b>: 'RNN'
* <b> datas </b>: ['data']
* <b> backgrounds </b>: ['ttbar', 'wtaunu', 'wmunu', 'wenu', 'ztautau', 'zmumu', 'zee', 'znunu', 'diboson', 'singletop', 'ttX']
* <b> signals </b>: gluinos -> ['GG_XXX_YYY'] & squarks -> ['SS_XXX_YYY']

<br>

<hr>

Run 2:
* <b> release </b>: 'R24'
* <b> analysis base </b>: '24.2.28'
* <b> campaigns </b>: ['MC20a', 'MC20d', 'MC20e']
* <b> architecture </b>: 'RNN'
* <b> datas </b>: ['data']
* <b> backgrounds </b>: ['ttbar', 'wtaunu', 'wmunu', 'wenu', 'ztautau', 'zmumu', 'zee', 'znunu', 'diboson', 'higgs', 'singletop', 'ttX']
* <b> signals </b>: gluinos -> ['GG_XXX_YYY'] & squarks -> ['SS_XXX_YYY']

<br>

<hr>

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, samples, features) are defined in `configs/` and can be overridden here.

In [4]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")
cfg

{'experiment_name': 'tau-supersymmetry-baseline', 'seed': 1, 'data': {'processed_path': 'data/processed', 'batch_size': 256, 'test_split': 0.2, 'padding_threshold': 3, 'nan_threshold': 0.5}, 'model': {'name': 'xgboost', 'booster': 'gbtree', 'n_estimators': 100, 'learning_rate': 0.5, 'max_depth': 5, 'min_child_weight': 1, 'gamma': 0, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_alpha': 0, 'reg_lambda': 0, 'max_delta_step': 0, 'tree_method': 'hist', 'device': 'cuda', 'seed': 1}, 'pipeline': {'name': 'standard_training', 'save_model': True, 'split_strategy': 'train_test', 'n_splits': 5, 'early_stopping_rounds': 50}, 'features': {'cleaning': ['eventClean', 'isBadTile', 'jet_isBadTight', 'IsMETTrigPassed', 'tau_JetRNNMedium'], 'truth': ['tau_isTruthMatchedTau', 'tau_truthType'], 'weights': ['mcEventWeight', 'pileupweight', 'beamSpotWeight', 'tau_medium_weight', 'ele_weight', 'mu_weight', 'jvt_weight', 'bjet_weight', 'jet_btag_weight', 'lumiweight'], 'tau': ['tau_n', 'tau_pt', 'tau_eta', 

### Analysis

Constructing the analysis configuration object. 

In [5]:
from src.processing.analysis import AnalysisConfig

analysis = AnalysisConfig(**cfg.analysis)
analysis

AnalysisConfig(run='Run2', release='R24', analysis_base='24.2.28', campaigns=['MC20a', 'MC20d', 'MC20e'], scope='ML', region='Preselection', channel='1', subject=None, sub_subject=None)

### Samples

Building sample lists from config.

In [6]:
from src.processing.analysis import resolve_samples

samples = resolve_samples(cfg)
samples

{'data': [Sample(id='data', label='data')],
 'background': [Sample(id='ttbar', label='$t \\bar{t}$'),
  Sample(id='wtaunu', label='$W \\rightarrow \\tau\\nu$'),
  Sample(id='wmunu', label='$W \\rightarrow \\mu\\nu$'),
  Sample(id='wenu', label='$W \\rightarrow e \\nu$'),
  Sample(id='ztautau', label='$Z \\rightarrow \\tau\\tau$'),
  Sample(id='zmumu', label='$Z \\rightarrow \\mu\\mu$'),
  Sample(id='zee', label='$Z \\rightarrow ee$'),
  Sample(id='znunu', label='$Z \\rightarrow \\nu\\nu$'),
  Sample(id='diboson', label='diboson'),
  Sample(id='singletop', label='singletop')],
 'signal': []}

## Processing

Reading ROOT ntuples, applying selection cuts (cleaning, channel, RNN, truth-matching, kinematic, region), and computing event weights. Each sample is processed across all campaigns and concatenated.

In [7]:
from src.processing.processor import process_samples

### Background

Monte Carlo-based backgrounds.

In [8]:
bg_raw = process_samples(cfg, sample_type="background", sample_ids=[s.id for s in samples["background"]])

FileNotFoundError: file not found

    '/disk/atlas3/data_MC/24.2.28/PHYS/MC20a/vector_XEplateau/ttbar.root'

Files may be specified as:
   * str/bytes: relative or absolute filesystem path or URL, without any colons
         other than Windows drive letter or URL schema.
         Examples: "rel/file.root", "C:\abs\file.root", "http://where/what.root"
   * str/bytes: same with an object-within-ROOT path, separated by a colon.
         Example: "rel/file.root:tdirectory/ttree"
   * pathlib.Path: always interpreted as a filesystem path or URL only (no
         object-within-ROOT path), regardless of whether there are any colons.
         Examples: Path("rel:/file.root"), Path("/abs/path:stuff.root")

Functions that accept many files (uproot.iterate, etc.) also allow:
   * glob syntax in str/bytes and pathlib.Path.
         Examples: Path("rel/*.root"), "/abs/*.root:tdirectory/ttree"
   * dict: keys are filesystem paths, values are objects-within-ROOT paths.
         Example: {"/data_v1/*

### Signal

Monte Carlo-based (gluinos and squarks) signals.

In [ ]:
signal_raw = (
    process_samples(cfg, sample_type="signal", sample_ids=[s.id for s in samples["signal"]])
    if samples["signal"]
    else {}
)

## Merging

Merging processed samples. Backgrounds are grouped by the merge strategy defined in `configs/merge/default.yaml` (`as_primary` groups into topquarks, wtaunu, ztautau, diboson). Class labels are assigned for ML training.

In [ ]:
from src.processing.merger import (
    merge_backgrounds,
    combine_background_signal,
    assign_class,
)

merged = merge_backgrounds(bg_raw, cfg)

if signal_raw:
    merged = combine_background_signal(merged, signal_raw)

assign_class(merged)

## Serialization

Saving each sample as a parquet file to the path derived from cfg (scope/analysis_base/run/region/channel). One file per sample key.

In [ ]:
from src.processing.io import save_samples
from src.processing.analysis import get_output_paths

samples_dir = path / get_output_paths(cfg)["samples_dir"]
save_samples(merged, samples_dir)